# RetailHero Data Understanding and Cleaning

## Objective

This notebook prepares the RetailHero data for downstream uplift-modeling experiments.

The modeling population is defined by `uplift_train`, which contains the customers with observed treatment assignment and purchase outcome. Customer information, historical purchases, and product metadata are used to describe this population before modeling.

The goal is to produce a clean and well-understood analytical dataset before exploratory data analysis and feature engineering.

## Scope

The workflow is:

```text
Raw RetailHero data
        ↓
Structural validation
        ↓
Define modeling population from uplift_train
        ↓
Inspect relevant customers, purchases, and products
        ↓
Investigate data-quality issues
        ↓
Apply cleaning rules
        ↓
Validate and save cleaned data
```

Raw tables are first checked for structural validity. Detailed data-quality analysis is then restricted to:

 - Customers contained in uplift_train;
 - Historical purchases belonging to those customers;
 - Products appearing in those purchase histories.

Cleaning decisions are based on observed data patterns and practical retail plausibility. Raw source files remain unchanged, while the cleaned analytical tables contain only the population required by this project.

### RetailHero Data Structure

The RetailHero data used in this experiment consists of four tables:

* `clients`: customer information;
* `products`: product metadata;
* `purchases`: customer purchase history recorded before the promotional communication;
* `uplift_train`: customers in the labeled uplift experiment, including treatment assignment and observed outcome.

The tables are connected through `client_id` and `product_id`.

```text
uplift_train
      │
      │ client_id
      ▼
   clients
      │
      │ client_id
      ▼
  purchases ─── product_id ───► products
  ```

The `uplift_train` defines the modeling population and provides:

  ```text
  treatment = treatment_flg
  outcome   = target
  ```

This project does not use the original competition `uplift_test`. The 200,039 labeled customers in `uplift_train` are used to construct the project dataset and are later split internally into train, validation, and locked-test sets.

In [384]:
from pathlib import Path

import duckdb
import pandas as pd

from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [385]:
TABLE_HIGHLIGHT = {
    "blue": {"background-color": "#1976D2", "color": "white", "font-weight": "700"},
    "red": {"background-color": "#D32F2F", "color": "white", "font-weight": "700"},
}


def style_table(df: pd.DataFrame, formats: dict | None = None):
    return df.style.hide(axis="index").format(formats or {}, na_rep="—")


def paint(styler, rows, columns, style_key: str):
    rows = list(rows)
    if not rows:
        return styler

    columns = [columns] if isinstance(columns, str) else list(columns)
    return styler.set_properties(
        subset=pd.IndexSlice[rows, columns],
        **TABLE_HIGHLIGHT[style_key],
    )

In [386]:
PROJECT_ROOT = Path.cwd().parents[1]

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "retailhero"
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim" / "retailhero"

RAW_FILES = {
    "clients": RAW_DATA_DIR / "clients.csv",
    "products": RAW_DATA_DIR / "products.csv",
    "purchases": RAW_DATA_DIR / "purchases.csv",
    "uplift_train": RAW_DATA_DIR / "uplift_train.csv",
}

INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)

missing_files = [f"{name}: {path}" for name, path in RAW_FILES.items() if not path.exists()]
if missing_files:
    raise FileNotFoundError("Missing RetailHero raw files:\n" + "\n".join(missing_files))

print("Project root:", PROJECT_ROOT)
print("Raw data:", RAW_DATA_DIR)
print("Interim data:", INTERIM_DATA_DIR)

Project root: d:\thao\d\uplif_model\uplif_customer_selection
Raw data: d:\thao\d\uplif_model\uplif_customer_selection\data\raw\retailhero
Interim data: d:\thao\d\uplif_model\uplif_customer_selection\data\interim\retailhero


In [387]:
file_inventory = pd.DataFrame([
    {
        "table": table_name,
        "file": path.name,
        "size_mb": path.stat().st_size / (1024**2),
    }
    for table_name, path in RAW_FILES.items()
]).sort_values("size_mb", ascending=False)

display(file_inventory)

,table,file,size_mb
2,purchases,purchases.csv,"4,256.9881"
0,clients,clients.csv,20.7293
1,products,products.csv,3.7101
3,uplift_train,uplift_train.csv,2.8616


## DuckDB Loading Strategy

The raw RetailHero files are intentionally **not loaded into pandas as full tables.**

This is particularly important for `purchases`, which is much larger than the other tables.

The notebook uses the following loading steps:

1. Keep the original CSV files unchanged under `data/raw/retailhero/`.
2. Use DuckDB to query the raw tables.
3. Convert the large `purchases.csv` file once into a local Parquet cache.
4. Query subsequent `purchases` data from the Parquet version through DuckDB.
5. Avoid loading the complete `purchases` table into pandas.
6. Convert only small query results, summaries, samples, or final customer-level feature tables to pandas.

This approach avoids repeatedly parsing the large CSV while allowing DuckDB to read only the required columns and rows.

The remaining RetailHero tables are much smaller and can remain as DuckDB views over their original CSV files.


In [388]:
DUCKDB_CACHE_DIR = INTERIM_DATA_DIR / "_duckdb_cache"
DUCKDB_TEMP_DIR = DUCKDB_CACHE_DIR / "tmp"

DUCKDB_CACHE_DIR.mkdir(parents=True, exist_ok=True)
DUCKDB_TEMP_DIR.mkdir(parents=True, exist_ok=True)

duckdb_connection = duckdb.connect(database=":memory:")

def to_sql_path(path: Path) -> str:
    """Return an absolute path escaped for SQL."""
    return path.resolve().as_posix().replace("'", "''")

duckdb_connection.execute(f"SET temp_directory = '{to_sql_path(DUCKDB_TEMP_DIR)}'")

In [389]:
PURCHASES_PARQUET_PATH = DUCKDB_CACHE_DIR / "purchases.parquet"
raw_purchases_path = RAW_FILES["purchases"]

cache_outdated = (
    not PURCHASES_PARQUET_PATH.exists()
    or raw_purchases_path.stat().st_mtime > PURCHASES_PARQUET_PATH.stat().st_mtime
)

if cache_outdated:
    print("Creating purchases Parquet cache...")
    duckdb_connection.execute(
        f"""
        COPY (
            SELECT *
            FROM read_csv('{to_sql_path(raw_purchases_path)}', header=true)
        )
        TO '{to_sql_path(PURCHASES_PARQUET_PATH)}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
        """
    )
else:
    print("Using existing purchases Parquet cache.")

Using existing purchases Parquet cache.


In [390]:
for table_name in ("clients", "products", "uplift_train"):
    path = RAW_FILES[table_name]

    duckdb_connection.execute(
        f"""
        CREATE OR REPLACE VIEW {table_name}_raw AS
        SELECT *
        FROM read_csv(
            '{to_sql_path(path)}',
            header=true,
            sample_size=-1,
            strict_mode=true
        )
        """
    )

duckdb_connection.execute(
    f"""
    CREATE OR REPLACE VIEW purchases_raw AS
    SELECT *
    FROM read_parquet('{to_sql_path(PURCHASES_PARQUET_PATH)}')
    """
)

RAW_RELATIONS = {
    "clients": "clients_raw",
    "products": "products_raw",
    "purchases": "purchases_raw",
    "uplift_train": "uplift_train_raw",
}

display(duckdb_connection.execute("SHOW TABLES").df())

,name
0,clients_raw
1,products_raw
2,purchases_raw
3,uplift_train_raw


## Data Overview

### Table and Column Description

The competition documentation describes the purpose of each table but does not provide a complete business definition for every raw field.

The descriptions below are therefore working interpretations based on the available documentation, column names, and observed table structure. Fields whose exact semantics remain unclear are kept explicit rather than assigned an unsupported business meaning.


### `clients`

One row is expected to represent one customer.

| Column              | Description                                                        |
| ------------------- | ------------------------------------------------------------------ |
| `client_id`         | Unique customer identifier used to link tables                     |
| `first_issue_date`  | Date associated with the customer's first loyalty-card issuance    |
| `first_redeem_date` | Date associated with the customer's first loyalty-point redemption |
| `age`               | Customer age                                                       |
| `gender`            | Customer gender                                                    |

### `products`

One row is expected to represent one product.

| Column             | Description                                             |
| ------------------ | ------------------------------------------------------- |
| `product_id`       | Unique product identifier used to link purchase records |
| `level_1`          | Highest-level product category                          |
| `level_2`          | Second-level product category                           |
| `level_3`          | Third-level product category                            |
| `level_4`          | Most detailed product category provided                 |
| `segment_id`       | Product segment identifier                              |
| `brand_id`         | Brand identifier                                        |
| `vendor_id`        | Vendor or supplier identifier                           |
| `netto`            | Numeric product attribute provided by RetailHero                                |
| `is_own_trademark` | Whether the product belongs to the retailer's own brand |
| `is_alcohol`       | Whether the product is alcoholic                        |

### `purchases`

The table contains customer purchase history recorded before the promotional communication.

The initial working assumption is that each row represents a product line within a transaction. This assumption is validated later because a transaction may contain multiple products and `transaction_id` is not assumed to be globally unique without verification.


| Column                    | Description                                                          |
| ------------------------- | -------------------------------------------------------------------- |
| `client_id`               | Customer associated with the transaction                             |
| `transaction_id`          | Transaction or receipt identifier                                    |
| `transaction_datetime`    | Date and time of the transaction                                     |
| `regular_points_received` | Regular loyalty points received                                      |
| `express_points_received` | Express loyalty points received                                      |
| `regular_points_spent`    | Regular loyalty points redeemed                                      |
| `express_points_spent`    | Express loyalty points redeemed                                      |
| `purchase_sum`            | Purchase amount recorded for the transaction                         |
| `store_id`                | Store identifier                                                     |
| `product_id`              | Purchased product identifier                                         |
| `product_quantity`        | Quantity of the product purchased                                    |
| `trn_sum_from_iss`        | Amount associated with the product line on the point-issuance side   |
| `trn_sum_from_red`        | Amount associated with the product line on the point-redemption side |

The exact business rules behind `express_points_*`, `trn_sum_from_iss`, and `trn_sum_from_red` are not documented in detail in the available competition description.


### `uplift_train`

One row is expected to represent one customer in the labeled uplift experiment.

| Column          | Description                                                                    |
| --------------- | ------------------------------------------------------------------------------ |
| `client_id`     | Customer identifier used to join with customer and purchase data               |
| `treatment_flg` | Treatment assignment: `1` if promotional communication was sent, otherwise `0` |
| `target`        | Binary outcome indicating whether the customer made a purchase afterward       |

For this experiment:

```text
treatment = treatment_flg
outcome   = target
```

The remaining tables provide information that can potentially be transformed into **pre-treatment customer features**, while `uplift_train` determines the treatment and observed outcome for each modeled customer.


In [391]:
table_shapes = []

for table_name, relation_name in RAW_RELATIONS.items():
    rows = duckdb_connection.execute(f"SELECT COUNT(*) FROM {relation_name}").fetchone()[0]
    columns = len(duckdb_connection.execute(f"DESCRIBE {relation_name}").fetchall())

    table_shapes.append({
        "table": table_name,
        "rows": rows,
        "columns": columns,
    })

display(pd.DataFrame(table_shapes))

,table,rows,columns
0,clients,400162,5
1,products,43038,11
2,purchases,45786568,13
3,uplift_train,200039,3


In [392]:
for table_name, relation_name in RAW_RELATIONS.items():
    print(f"\n{table_name}")
    display(duckdb_connection.execute(f"DESCRIBE {relation_name}").df())


clients


,column_name,column_type,null,key,default,extra
0,client_id,VARCHAR,YES,None,None,None
1,first_issue_date,TIMESTAMP,YES,None,None,None
2,first_redeem_date,TIMESTAMP,YES,None,None,None
3,age,BIGINT,YES,None,None,None
4,gender,VARCHAR,YES,None,None,None



products


,column_name,column_type,null,key,default,extra
0,product_id,VARCHAR,YES,None,None,None
1,level_1,VARCHAR,YES,None,None,None
2,level_2,VARCHAR,YES,None,None,None
3,level_3,VARCHAR,YES,None,None,None
4,level_4,VARCHAR,YES,None,None,None
5,segment_id,DOUBLE,YES,None,None,None
6,brand_id,VARCHAR,YES,None,None,None
7,vendor_id,VARCHAR,YES,None,None,None
8,netto,DOUBLE,YES,None,None,None
9,is_own_trademark,BIGINT,YES,None,None,None



purchases


,column_name,column_type,null,key,default,extra
0,client_id,VARCHAR,YES,None,None,None
1,transaction_id,VARCHAR,YES,None,None,None
2,transaction_datetime,TIMESTAMP,YES,None,None,None
3,regular_points_received,DOUBLE,YES,None,None,None
4,express_points_received,DOUBLE,YES,None,None,None
5,regular_points_spent,DOUBLE,YES,None,None,None
6,express_points_spent,DOUBLE,YES,None,None,None
7,purchase_sum,DOUBLE,YES,None,None,None
8,store_id,VARCHAR,YES,None,None,None
9,product_id,VARCHAR,YES,None,None,None



uplift_train


,column_name,column_type,null,key,default,extra
0,client_id,VARCHAR,YES,None,None,None
1,treatment_flg,BIGINT,YES,None,None,None
2,target,BIGINT,YES,None,None,None


In [393]:
for table_name, relation_name in RAW_RELATIONS.items():
    print(f"\n{table_name}")
    display(duckdb_connection.execute(f"SELECT * FROM {relation_name} LIMIT 5").df())


clients


,client_id,first_issue_date,first_redeem_date,age,gender
0,000012768d,2017-08-05 15:40:48,2018-01-04 19:30:07,45,U
1,000036f903,2017-04-10 13:54:23,2017-04-23 12:37:56,72,F
2,000048b7a6,2018-12-15 13:33:11,NaT,68,F
3,000073194a,2017-05-23 12:56:14,2017-11-24 11:18:01,60,F
4,00007c7133,2017-05-22 16:17:08,2018-12-31 17:17:33,67,U



products


,product_id,level_1,level_2,level_3,level_4,segment_id,brand_id,vendor_id,netto,is_own_trademark,is_alcohol
0,0003020d3c,c3d3a8e8c6,c2a3ea8d5e,b7cda0ec0c,6376f2a852,123.0000,394a54a7c1,9eaff48661,0.4000,0,0
1,0003870676,e344ab2e71,52f13dac0c,d3cfe81323,6dc544533f,105.0000,acd3dd483f,10486c3cf0,0.6800,0,0
2,0003ceaf69,c3d3a8e8c6,f2333c90fb,419bc5b424,f6148afbc0,271.0000,f597581079,764e660dda,0.5000,0,0
3,000701e093,ec62ce61e3,4202626fcb,88a515c084,48cf3d488f,172.0000,54a90fe769,03c2d70bad,0.1120,0,0
4,0007149564,e344ab2e71,52f13dac0c,d3cfe81323,6dc544533f,105.0000,63417fe1f3,f329130198,0.6000,0,0



purchases


,client_id,transaction_id,transaction_datetime,regular_points_received,express_points_received,regular_points_spent,express_points_spent,purchase_sum,store_id,product_id,product_quantity,trn_sum_from_iss,trn_sum_from_red
0,000012768d,7e3e2e3984,2018-12-01 07:12:45,10.0000,0.0000,0.0000,0.0000,"1,007.0000",54a4a11a29,9a80204f78,2.0000,80.0000,NaN
1,000012768d,7e3e2e3984,2018-12-01 07:12:45,10.0000,0.0000,0.0000,0.0000,"1,007.0000",54a4a11a29,da89ebd374,1.0000,65.0000,NaN
2,000012768d,7e3e2e3984,2018-12-01 07:12:45,10.0000,0.0000,0.0000,0.0000,"1,007.0000",54a4a11a29,0a95e1151d,1.0000,24.0000,NaN
3,000012768d,7e3e2e3984,2018-12-01 07:12:45,10.0000,0.0000,0.0000,0.0000,"1,007.0000",54a4a11a29,4055b15e4a,2.0000,50.0000,NaN
4,000012768d,7e3e2e3984,2018-12-01 07:12:45,10.0000,0.0000,0.0000,0.0000,"1,007.0000",54a4a11a29,a685f1916b,1.0000,22.0000,NaN



uplift_train


,client_id,treatment_flg,target
0,000012768d,0,1
1,000036f903,1,1
2,00010925a5,1,1
3,0001f552b0,1,1
4,00020e7b18,1,1


## Raw Structural Validation

Before defining the modeling population, the raw tables are checked for basic structural integrity:

1. Primary identifiers are non-null and unique where expected;
2. `treatment_flg` and `target` contain valid binary values;
3. Table relationships do not contain orphan customer or product identifiers.

Value distributions, missingness, and domain-specific data quality are analyzed only after the modeling population is defined.

In [394]:
validation_issues = []

UNIQUE_KEYS = {
    "clients": "client_id",
    "products": "product_id",
    "uplift_train": "client_id",
}

key_profiles = []

for table_name, key in UNIQUE_KEYS.items():
    rows, null_keys, distinct_keys = duckdb_connection.execute(
        f""" 
        SELECT
            COUNT(*),
            COUNT(*) FILTER (WHERE {key} IS NULL),
            COUNT(DISTINCT {key})
        FROM {RAW_RELATIONS[table_name]}
        """
    ).fetchone()

    duplicate_keys = rows - null_keys - distinct_keys

    key_profiles.append({
        "table": table_name,
        "rows": rows,
        "distinct_keys": distinct_keys,
        "null_keys": null_keys,
        "duplicate_keys": duplicate_keys,
    })

    if null_keys or duplicate_keys:
        validation_issues.append(
            f"{table_name}.{key}: {null_keys} null keys, {duplicate_keys} duplicate keys"
    )

display(pd.DataFrame(key_profiles))

,table,rows,distinct_keys,null_keys,duplicate_keys
0,clients,400162,400162,0,0
1,products,43038,43038,0,0
2,uplift_train,200039,200039,0,0


In [395]:
FULL_ROW_DUPLICATE_TABLES = [
    "clients",
    "products",
    "uplift_train",
]

full_row_duplicate_profiles = []

for table_name in FULL_ROW_DUPLICATE_TABLES:
    relation_name = RAW_RELATIONS[table_name]

    row_counts = duckdb_connection.execute(
        f"""
        WITH distinct_rows AS (
            SELECT DISTINCT *
            FROM {relation_name}
        )

        SELECT
            (SELECT COUNT(*) FROM {relation_name}) AS rows,
            (SELECT COUNT(*) FROM distinct_rows) AS distinct_rows
        """
    ).fetchone()

    rows, distinct_rows = row_counts
    duplicate_rows = rows - distinct_rows

    full_row_duplicate_profiles.append({
        "table": table_name,
        "rows": rows,
        "distinct_rows": distinct_rows,
        "duplicate_rows": duplicate_rows,
    })

    if duplicate_rows:
        validation_issues.append(
            f"{table_name}: {duplicate_rows} duplicate full rows"
        )

full_row_duplicate_profile = pd.DataFrame(full_row_duplicate_profiles)

display(full_row_duplicate_profile)


,table,rows,distinct_rows,duplicate_rows
0,clients,400162,400162,0
1,products,43038,43038,0
2,uplift_train,200039,200039,0


In [396]:
treatment_target_check = duckdb_connection.execute(
    """
    SELECT 
        COUNT(*) FILTER (WHERE treatment_flg IS NULL) AS treatment_nulls,
        COUNT(*) FILTER(
            WHERE treatment_flg IS NOT NULL
                AND treatment_flg NOT IN (0, 1)
        ) AS treatment_invalid,
        COUNT(*) FILTER (WHERE target IS NULL) AS target_nulls,
        COUNT(*) FILTER (
            WHERE target IS NOT NULL
                AND target NOT IN (0, 1)
        ) AS target_invalid
    FROM uplift_train_raw
    """
).df()

display(treatment_target_check)

if treatment_target_check.iloc[0].sum() > 0:
    validation_issues.append("uplift_train: invalid or missing treatment/target values")

treatment_target_profile = duckdb_connection.execute(
    """
    SELECT
        treatment_flg,
        target,
        COUNT(*) AS customers,
        COUNT(*) * 1.0 / SUM(COUNT(*)) OVER () AS proportion
    FROM uplift_train_raw
    GROUP BY treatment_flg, target
    ORDER BY treatment_flg, target
    """
).df()

display(treatment_target_profile)


,treatment_nulls,treatment_invalid,target_nulls,target_invalid
0,0,0,0,0


,treatment_flg,target,customers,proportion
0,0,0,39695,0.1984
1,0,1,60363,0.3018
2,1,0,36342,0.1817
3,1,1,63639,0.3181


In [397]:
uplift_relationship_check = duckdb_connection.execute(
    """
    SELECT COUNT(*) FILTER (WHERE c.client_id IS NULL)
        AS uplift_clients_missing_from_clients
    FROM uplift_train_raw AS u
    LEFT JOIN clients_raw AS c USING (client_id)
    """
).df()

purchase_relationship_check = duckdb_connection.execute(
    """
    SELECT
        COUNT(*) FILTER (WHERE c.client_id IS NULL) AS rows_with_unknown_client,
        COUNT(*) FILTER (WHERE pr.product_id IS NULL) AS rows_with_unknown_product
    FROM purchases_raw AS p
    LEFT JOIN clients_raw AS c USING (client_id)
    LEFT JOIN products_raw AS pr USING (product_id)
    """
).df()

display(uplift_relationship_check)
display(purchase_relationship_check)

if uplift_relationship_check.iloc[0, 0] > 0:
    validation_issues.append("uplift_train contains unknown client_id values")

if purchase_relationship_check.iloc[0].sum() > 0:
    validation_issues.append("purchases contains unknown client_id or product_id values")

,uplift_clients_missing_from_clients
0,0


,rows_with_unknown_client,rows_with_unknown_product
0,0,0


In [398]:
duckdb_connection.execute(
    """
    CREATE OR REPLACE VIEW clients_model AS
    SELECT c.*
    FROM clients_raw AS c
    INNER JOIN uplift_train_raw AS u USING (client_id)
    """
)

duckdb_connection.execute(
    """
    CREATE OR REPLACE VIEW purchases_model AS
    SELECT p.*
    FROM purchases_raw AS p
    INNER JOIN uplift_train_raw AS u USING (client_id)
    """
)

duckdb_connection.execute(
    """
    CREATE OR REPLACE VIEW products_model AS
    SELECT pr.*
    FROM products_raw AS pr
    WHERE pr.product_id IN (
        SELECT DISTINCT product_id
        FROM purchases_model
    )
    """
)

MODEL_RELATIONS = {
    "clients": "clients_model",
    "products": "products_model",
    "purchases": "purchases_model",
    "uplift_train": "uplift_train_raw",
}

scope_profile = []

for table_name, relation_name in MODEL_RELATIONS.items():
    rows = duckdb_connection.execute(f"SELECT COUNT(*) FROM {relation_name}").fetchone()[0]
    scope_profile.append({"table": table_name, "rows": rows})

display(pd.DataFrame(scope_profile))

,table,rows
0,clients,200039
1,products,40716
2,purchases,22882690
3,uplift_train,200039


In [399]:
NUMERIC_COLUMNS = {
    "clients": ["age"],
    "products": ["netto"],
    "purchases": [
        "regular_points_received",
        "express_points_received",
        "regular_points_spent",
        "express_points_spent",
        "purchase_sum",
        "product_quantity",
        "trn_sum_from_iss",
        "trn_sum_from_red",
    ],
}

CATEGORICAL_COLUMNS = {
    "clients": ["gender"],
    "products": [
        "level_1",
        "level_2",
        "level_3",
        "level_4",
        "segment_id",
        "brand_id",
        "vendor_id",
    ],
    "purchases": ["store_id"],
}

DATETIME_COLUMNS = {
    "clients": ["first_issue_date", "first_redeem_date"],
    "purchases": ["transaction_datetime"],
}

REQUIRED_NON_NULL = {
    "clients": {"client_id"},
    "products": {"product_id"},
    "purchases": {"client_id", "transaction_id", "transaction_datetime", "product_id"},
    "uplift_train": {"client_id", "treatment_flg", "target"},
}

In [400]:
NUMERIC_AGGREGATES = {
    "count": "COUNT({column})",
    "mean": "AVG({column})",
    "std": "STDDEV_SAMP({column})",
    "min": "MIN({column})",
    "25%": "APPROX_QUANTILE({column}, 0.25)",
    "50%": "APPROX_QUANTILE({column}, 0.50)",
    "75%": "APPROX_QUANTILE({column}, 0.75)",
    "max": "MAX({column})",
}


def numeric_profile(table_name: str, columns: list[str]) -> pd.DataFrame:
    expressions = [
        f'{expression.format(column=f"""\"{column}\"""")} AS "{column}__{metric}"'
        for column in columns
        for metric, expression in NUMERIC_AGGREGATES.items()
    ]

    result = duckdb_connection.execute(
        f'SELECT {", ".join(expressions)} FROM {MODEL_RELATIONS[table_name]}'
    ).df().iloc[0]

    return pd.DataFrame([
        {
            "table": table_name,
            "column": column,
            **{metric: result[f"{column}__{metric}"] for metric in NUMERIC_AGGREGATES},
        }
        for column in columns
    ])


numeric_profiles = pd.concat(
    [numeric_profile(table, columns) for table, columns in NUMERIC_COLUMNS.items()],
    ignore_index=True,
)

display(numeric_profiles)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,table,column,count,mean,std,min,25%,50%,75%,max
0,clients,age,"200,039.0000",46.4173,49.5325,"-7,491.0000",33.0000,45.0000,59.0000,"1,852.0000"
1,products,netto,"40,713.0000",0.5441,8.5059,0.0000,0.1506,0.3054,0.5000,"1,150.0000"
2,purchases,regular_points_received,"22,882,690.0000",8.0428,12.8138,0.0000,1.3744,3.6933,10.3166,"2,399.0000"
3,purchases,express_points_received,"22,882,690.0000",0.0609,2.4240,0.0000,0.0000,0.0000,0.0000,300.0000
4,purchases,regular_points_spent,"22,882,690.0000",-5.2990,36.2238,"-5,066.0000",0.0000,0.0000,0.0000,0.0000
5,purchases,express_points_spent,"22,882,690.0000",-0.3183,3.2901,-300.0000,0.0000,0.0000,0.0000,0.0000
6,purchases,purchase_sum,"22,882,690.0000",775.8724,795.2956,0.0000,285.4499,538.2191,975.6385,"29,611.4800"
7,purchases,product_quantity,"22,882,690.0000",1.2460,1.1019,0.0000,1.0000,1.0000,1.0000,648.0000
8,purchases,trn_sum_from_iss,"22,882,690.0000",73.4216,86.6362,0.0000,29.9483,51.1077,89.8894,"29,074.0000"
9,purchases,trn_sum_from_red,"1,513,397.0000",76.7815,84.5437,0.0000,31.1804,54.5120,94.0574,"6,400.0000"


In [401]:
missing_profiles = []

for table_name, relation_name in MODEL_RELATIONS.items():
    columns = [row[0] for row in duckdb_connection.execute(
        f"DESCRIBE {relation_name}"
    ).fetchall()]

    expressions = [
        f'COUNT(*) FILTER (WHERE "{column}" IS NULL) AS "{column}"'
        for column in columns
    ]

    result = duckdb_connection.execute(
        f'SELECT COUNT(*) AS total_rows, {", ".join(expressions)} FROM {relation_name}'
    ).df().iloc[0]

    total_rows = int(result["total_rows"])

    for column in columns:
        null_count = int(result[column])
        required = column in REQUIRED_NON_NULL.get(table_name, set())

        missing_profiles.append({
            "table": table_name,
            "column": column,
            "null_count": null_count,
            "null_rate": null_count / total_rows,
            "required_non_null": required,
        })

        if required and null_count:
            validation_issues.append(
                f"{table_name}.{column}: {null_count} unexpected null values"
            )

missing_profile = pd.DataFrame(missing_profiles)

display(
    missing_profile.sort_values(
        ["null_rate", "table", "column"],
        ascending=[False, True, True],
    )
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,table,column,null_count,null_rate,required_non_null
28,purchases,trn_sum_from_red,21369293,0.9339,False
11,products,brand_id,4767,0.1171,False
2,clients,first_redeem_date,17546,0.0877,False
10,products,segment_id,1457,0.0358,False
12,products,vendor_id,26,0.0006,False
6,products,level_1,3,0.0001,False
7,products,level_2,3,0.0001,False
8,products,level_3,3,0.0001,False
9,products,level_4,3,0.0001,False
13,products,netto,3,0.0001,False


In [402]:
MAX_FULL_CATEGORIES = 20
TOP_CATEGORIES = 10

categorical_profiles = []
categorical_counts = {}

for table_name, columns in CATEGORICAL_COLUMNS.items():
    relation_name = MODEL_RELATIONS[table_name]

    for column in columns:
        n_unique = duckdb_connection.execute(
            f"""
            SELECT COUNT(DISTINCT "{column}")
            FROM {relation_name}
            WHERE "{column}" IS NOT NULL
            """
        ).fetchone()[0]

        limit = "" if n_unique <= MAX_FULL_CATEGORIES else f"LIMIT {TOP_CATEGORIES}"

        counts = duckdb_connection.execute(
            f"""
            SELECT "{column}" AS value, COUNT(*) AS count
            FROM {relation_name}
            WHERE "{column}" IS NOT NULL
            GROUP BY "{column}"
            ORDER BY count DESC
            {limit}
            """
        ).df()
        
        categorical_profiles.append({
            "table": table_name,
            "column": column,
            "n_unique": n_unique,
        })

        categorical_counts[(table_name, column)] = counts

display(pd.DataFrame(categorical_profiles))

for (table_name, column), counts in categorical_counts.items():
    print(f"\n{table_name}.{column}")
    display(counts)

,table,column,n_unique
0,clients,gender,3
1,products,level_1,3
2,products,level_2,42
3,products,level_3,201
4,products,level_4,783
5,products,segment_id,116
6,products,brand_id,4183
7,products,vendor_id,3136
8,purchases,store_id,13879



clients.gender


,value,count
0,U,92832
1,F,73696
2,M,33511



products.level_1


,value,count
0,e344ab2e71,21038
1,c3d3a8e8c6,15799
2,ec62ce61e3,3876


...

,value,count
0,0d6f137fb6,3999
1,4da2dc345f,2906
2,b06ace74de,349
3,ab230258e9,265
4,63ba6b7a61,132
5,a548b9f2b8,112
6,7a282015f3,109
7,74251e93eb,106
8,8188d00160,105
9,aa73f98d68,104



products.vendor_id


,value,count
0,43acd80c1a,1467
1,63243765ed,338
2,4f276b13c1,320
3,c4e167b91e,318
4,83f98e6dc3,308
5,e6af81215a,286
6,addfbe3485,216
7,41aa9501e1,216
8,3034fb4c4a,210
9,ef3b92f068,181



purchases.store_id


,value,count
0,1f41964607,10100
1,cfbbd53ab7,9659
2,d25b798f30,9134
3,ed388527a1,9016
4,e3bf88fabf,8834
5,94bbb693dc,8425
6,a87bebd240,8274
7,f7390207ef,8244
8,7d467b36b2,8105
9,fac91d76e3,8056


In [403]:
product_binary_check = duckdb_connection.execute(
    """
    SELECT
        COUNT(*) FILTER (
            WHERE is_own_trademark IS NOT NULL
              AND is_own_trademark NOT IN (0, 1)
        ) AS invalid_is_own_trademark,
        COUNT(*) FILTER (
            WHERE is_alcohol IS NOT NULL
              AND is_alcohol NOT IN (0, 1)
        ) AS invalid_is_alcohol
    FROM products_model
    """
).df()

display(product_binary_check)

if product_binary_check.iloc[0].sum() > 0:
    validation_issues.append("products contains invalid binary values")

,invalid_is_own_trademark,invalid_is_alcohol
0,0,0


In [404]:
client_range_checks = duckdb_connection.execute(
    """
    SELECT
        COUNT(*) FILTER (WHERE age < 13) AS age_below_13,
        COUNT(*) FILTER (WHERE age > 100) AS age_above_100
    FROM clients_model
    """
).df()

product_range_checks = duckdb_connection.execute(
    """
    SELECT COUNT(*) FILTER (WHERE netto < 0) AS negative_netto
    FROM products_model
    """
).df()

display(client_range_checks)
display(product_range_checks)

,age_below_13,age_above_100
0,286,521


,negative_netto
0,0


In [405]:
datetime_profiles = []

for table_name, columns in DATETIME_COLUMNS.items():
    relation_name = MODEL_RELATIONS[table_name]

    for column in columns:
        min_date, max_date = duckdb_connection.execute(
            f'SELECT MIN("{column}"), MAX("{column}") FROM {relation_name}'
        ).fetchone()

        datetime_profiles.append({
            "table": table_name,
            "column": column,
            "min": min_date,
            "max": max_date,
        })

display(pd.DataFrame(datetime_profiles))

,table,column,min,max
0,clients,first_issue_date,2017-04-04 18:24:18,2019-03-15 21:44:14
1,clients,first_redeem_date,2017-04-11 09:42:20,2019-11-20 01:14:10
2,purchases,transaction_datetime,2018-11-21 21:02:33,2019-03-18 23:19:28


In [406]:
client_date_check = duckdb_connection.execute(
    """
    SELECT COUNT(*) FILTER (
        WHERE first_redeem_date IS NOT NULL
          AND first_redeem_date < first_issue_date
    ) AS redeem_before_issue
    FROM clients_model
    """
).df()

display(client_date_check)

,redeem_before_issue
0,245


## Modeling-Scope Data Quality Summary

Detailed quality analysis is performed on the population relevant to this experiment:

* **200,039 customers** from `uplift_train`
* **22,882,690 purchase rows** belonging to these customers
* **40,716 products** appearing in their purchase history

The raw structural checks found no duplicate primary identifiers, invalid treatment or target values, or orphan customer/product relationships.

Within the modeling population:

* `age` contains implausible values, with values ranging from `-7,491` to `1,852`
* `netto` is strongly right-skewed, with a median around `0.305` and a maximum of `1,150`
* `product_quantity` is usually small, with a median of `1` and a maximum of `648`
* `trn_sum_from_red` is highly sparse, with approximately `93.39%` missing values
* `brand_id` has approximately `11.71%` missing values
* `first_redeem_date` has approximately `8.77%` missing values
* `segment_id` has approximately `3.58%` missing values
* The remaining product metadata fields contain only small amounts of missing data

The binary product fields `is_own_trademark` and `is_alcohol` contain only `{0, 1}`.

Datetime coverage is:

* `first_issue_date`: `2017-04-04` to `2019-03-15`
* `first_redeem_date`: `2017-04-11` to `2019-11-20`
* `transaction_datetime`: `2018-11-21` to `2019-03-18`

There are **245 customers** where `first_redeem_date < first_issue_date`. These cases require closer inspection before defining the cleaning rule.

The next sections investigate only issues that can materially affect the cleaned analytical data.

## Investigation of Data-Quality Issues

Potential quality issues are investigated before applying a cleaning rule.

For each issue, the analysis checks:

1. How common the pattern is
2. Whether it is isolated or systematic
3. Whether it is plausible under normal retail behavior
4. Whether the original value can be trusted, corrected, or should be treated as missing

The main issues are customer age, customer date consistency, product `netto`, purchase quantities, transaction grain, and purchase-related numeric fields.

### Customer Age

`age` represents customer age in years.

RetailHero does not provide an official valid-age constraint. For this project, `13–100` is used as a practical analytical range based on normal retail customer plausibility and the observed distribution.

The investigation separates:

* `13–100`: plausible analytical range;
* `< 13`: implausibly young for the modeled retail customer population;
* `101–120`: unusually high;
* `< 0` or `> 120`: clearly invalid.

The distribution is examined before applying the final cleaning rule.

In [407]:
age_band_profile = duckdb_connection.execute(
    """
    SELECT
        CASE
            WHEN age < 0 THEN 'Negative age'
            WHEN age < 13 THEN 'Age 0-12'
            WHEN age <= 100 THEN 'Age 13-100'
            WHEN age <= 120 THEN 'Age 101-120'
            ELSE 'Age >120'
        END AS age_group,
        COUNT(*) AS clients,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct_clients,
        MIN(age) AS min_age,
        MAX(age) AS max_age
    FROM clients_model
    GROUP BY 1
    ORDER BY 1
    """
).df()

display(age_band_profile)

,age_group,clients,pct_clients,min_age,max_age
0,Age 0-12,230,0.1150,0,12
1,Age 101-120,409,0.2045,102,119
2,Age 13-100,199232,99.5966,13,100
3,Age >120,112,0.0560,123,1852
4,Negative age,56,0.0280,-7491,-1


In [408]:
suspicious_age_counts = duckdb_connection.execute(
    """
    SELECT age, COUNT(*) AS clients
    FROM clients_model
    WHERE age < 13 OR age > 100
    GROUP BY age
    ORDER BY clients DESC, age
    """
).df()

print("Distinct suspicious age values:", suspicious_age_counts["age"].nunique())
print("Customers with suspicious age values:", suspicious_age_counts["clients"].sum())

age_display = suspicious_age_counts.head(30)
styled = style_table(age_display, {"age": "{:,.0f}", "clients": "{:,.0f}"})

implausible_idx = age_display.index[
    (age_display["age"] < 0) | (age_display["age"] > 120)
]

display(paint(styled, implausible_idx, "age", "red"))

Distinct suspicious age values: 143
Customers with suspicious age values: 807


age,clients
115,210
119,184
12,42
11,33
1,28
10,28
9,23
0,21
8,14
2,11


In [409]:
age_quality_summary = duckdb_connection.execute(
    """
    SELECT
        COUNT(*) AS customers,
        COUNT(*) FILTER (WHERE age < 13 OR age > 100) AS suspicious_age_customers,
        ROUND(
            100.0 * COUNT(*) FILTER (WHERE age < 13 OR age > 100) / COUNT(*),
            4
        ) AS suspicious_age_rate
    FROM clients_model
    """
).df()

display(age_quality_summary)

,customers,suspicious_age_customers,suspicious_age_rate
0,200039,807,0.4034


### Age Quality

Among the **200,039 modeling customers**, **199,232 (99.60%)** have an age between `13–100`.

The remaining **807 customers (0.40%)** fall outside this range:

* `230` customers are between `0–12`;
* `409` customers are between `101–120`;
* `112` customers are above `120`;
* `56` customers have negative ages.

The full age range extends from `-7,491` to `1,852`, and the suspicious values include repeated extreme values rather than a single isolated typo.

Because the true ages cannot be reconstructed reliably, values outside `13–100` treated as unreliable rather than attempting to correct them.

### Current Conclusion

* Keep `age` when `13 <= age <= 100`.
* Convert `age < 13` or `age > 100` to `NULL`.
* Keep the customer record and all other customer information unchanged.

The `13–100` range is a project-level analytical rule based on retail plausibility and the observed data, not an official RetailHero constraint.

### Customer Date Consistency

`first_issue_date` is interpreted as the customer's first loyalty-card issuance timestamp, while `first_redeem_date` records the first observed redemption timestamp.

The expected ordering is:

`first_issue_date <= first_redeem_date`

Within the modeling population, **245 customers** violate this ordering.

The analysis therefore measures the timestamp difference in seconds and separates small same-day ordering inconsistencies from larger cross-day contradictions.

In [410]:
date_relation_profile = duckdb_connection.execute(
    """
    SELECT
        CASE
            WHEN first_redeem_date IS NULL THEN 'Missing first_redeem_date'
            WHEN first_redeem_date < first_issue_date THEN 'Redeem before issue'
            ELSE 'Expected ordering'
        END AS date_status,
        COUNT(*) AS clients,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct_clients
    FROM clients_model
    GROUP BY 1
    ORDER BY clients DESC
    """
).df()

display(date_relation_profile)

,date_status,clients,pct_clients
0,Expected ordering,182248,91.1062
1,Missing first_redeem_date,17546,8.7713
2,Redeem before issue,245,0.1225


In [411]:
redeem_before_issue_summary = duckdb_connection.execute(
    """
    WITH inconsistent_dates AS (
        SELECT
            date_diff('second', first_redeem_date, first_issue_date)
                AS seconds_before_issue
        FROM clients_model
        WHERE first_redeem_date < first_issue_date
    )
    SELECT
        COUNT(*) AS clients,
        MIN(seconds_before_issue) AS min_seconds,
        quantile_cont(seconds_before_issue, 0.50) AS median_seconds,
        quantile_cont(seconds_before_issue, 0.95) AS p95_seconds,
        MAX(seconds_before_issue) AS max_seconds
    FROM inconsistent_dates
    """
).df()

display(redeem_before_issue_summary)

,clients,min_seconds,median_seconds,p95_seconds,max_seconds
0,245,1,12.0000,43.0000,3701689


In [412]:
redeem_before_issue_bands = duckdb_connection.execute(
    """
    SELECT
        CASE
            WHEN CAST(first_redeem_date AS DATE) = CAST(first_issue_date AS DATE)
                THEN 'Same-day inversion'
            ELSE 'Cross-day inversion'
        END AS difference_group,
        COUNT(*) AS clients,
        MAX(date_diff('second', first_redeem_date, first_issue_date))
            AS max_seconds_before_issue
    FROM clients_model
    WHERE first_redeem_date < first_issue_date
    GROUP BY 1
    ORDER BY clients DESC
    """
).df()

display(redeem_before_issue_bands)

,difference_group,clients,max_seconds_before_issue
0,Same-day inversion,244,321
1,Cross-day inversion,1,3701689


In [413]:
redeem_before_issue_sample = duckdb_connection.execute(
    """
    SELECT
        client_id,
        first_issue_date,
        first_redeem_date,
        date_diff('second', first_redeem_date, first_issue_date)
            AS seconds_before_issue
    FROM clients_model
    WHERE first_redeem_date < first_issue_date
    ORDER BY seconds_before_issue DESC
    LIMIT 20
    """
).df()

display(
    style_table(
        redeem_before_issue_sample,
        {"seconds_before_issue": "{:,.0f}"},
    )
)

client_id,first_issue_date,first_redeem_date,seconds_before_issue
01440dcc9f,2018-11-09 13:50:09,2018-09-27 17:35:20,"3,701,689"
379ee75556,2018-09-23 23:37:21,2018-09-23 23:32:00,321
a1e30dace1,2018-12-21 16:17:18,2018-12-21 16:12:19,299
64bf2981e6,2019-02-06 16:49:55,2019-02-06 16:45:01,294
b307ca85a1,2019-01-31 19:23:10,2019-01-31 19:20:16,174
2ba4aaa1be,2018-07-23 20:08:47,2018-07-23 20:07:35,72
32ba83672c,2018-12-10 14:13:37,2018-12-10 14:12:36,61
37addc2e65,2019-03-08 13:04:52,2019-03-08 13:03:52,60
345c70d742,2018-12-08 13:30:26,2018-12-08 13:29:30,56
9617402b16,2018-12-05 14:58:20,2018-12-05 14:57:34,46


In [414]:
date_cleaning_counts = duckdb_connection.execute(
    """
    SELECT
        COUNT(*) FILTER (
            WHERE CAST(first_redeem_date AS DATE) = CAST(first_issue_date AS DATE)
        ) AS same_day_inversions,
        COUNT(*) FILTER (
            WHERE CAST(first_redeem_date AS DATE) < CAST(first_issue_date AS DATE)
        ) AS cross_day_inversions
    FROM clients_model
    WHERE first_redeem_date < first_issue_date
    """
).df()

display(date_cleaning_counts)

,same_day_inversions,cross_day_inversions
0,244,1


### Date Relationship Check

Among the **200,039 modeling customers**:

* **182,248 (91.11%)** have the expected date ordering
* **17,546 (8.77%)** have a missing `first_redeem_date`
* **245 (0.12%)** have `first_redeem_date < first_issue_date`

The 245 inconsistent records split into two clear groups:

* **244 same-day inversions**: the median difference is `12` seconds, the 95th percentile is `43` seconds, and the maximum difference is `321` seconds
* **1 cross-day inversion**: the redeem timestamp occurs about 43 days before the issue timestamp.

The same-day cases are small timestamp timing differences and are treated as recording issues rather than meaningful chronological differences.

### Current Conclusion

* For the **244 same-day inversions**, normalize `first_redeem_date` to `first_issue_date`.
* For the **single cross-day inconsistency**, keep `first_issue_date` and set only `first_redeem_date` to `NULL`.
* Existing missing `first_redeem_date` values remain missing.
* No customer is removed because of a date inconsistency.

### Product `netto`

`netto` is a numeric product attribute, but its exact meaning and unit are not documented in the available data description.

Within the products relevant to the modeling population:

* No negative values are observed
* The median is around `0.30`
* The distribution is strongly right-skewed
* Only three products have missing `netto`

Without knowing what the field represents, unusually large values cannot be reliably interpreted as valid values or data errors.

Therefore, `netto` is kept in the cleaned product table for data preservation, but it is not used for customer feature construction.

In [415]:
netto_distribution = duckdb_connection.execute(
    """
    SELECT
        COUNT(*) AS products,
        COUNT(netto) AS non_null_products,
        COUNT(*) FILTER (WHERE netto IS NULL) AS missing_products,
        MIN(netto) AS min,
        quantile_cont(netto, 0.25) AS p25,
        quantile_cont(netto, 0.50) AS median,
        quantile_cont(netto, 0.75) AS p75,
        quantile_cont(netto, 0.95) AS p95,
        quantile_cont(netto, 0.99) AS p99,
        quantile_cont(netto, 0.999) AS p999,
        MAX(netto) AS max
    FROM products_model
    """
).df()

display(netto_distribution)

,products,non_null_products,missing_products,min,p25,median,p75,p95,p99,p999,max
0,40716,40713,3,0.0000,0.1500,0.3000,0.5000,1.0000,1.5644,5.0000,"1,150.0000"


In [416]:
missing_netto_products = duckdb_connection.execute(
    """
    SELECT
        product_id,
        level_1,
        level_2,
        level_3,
        level_4,
        segment_id,
        brand_id,
        vendor_id
    FROM products_model
    WHERE netto IS NULL
    """
).df()

display(missing_netto_products)

,product_id,level_1,level_2,level_3,level_4,segment_id,brand_id,vendor_id
0,04d86b4b50,None,None,None,None,NaN,None,None
1,48cc0e256d,None,None,None,None,NaN,None,None
2,6a3d708544,None,None,None,None,NaN,None,None


In [417]:
netto_imputation_candidates = duckdb_connection.execute(
    """
    SELECT
        m.product_id,
        m.level_2,
        m.level_3,
        m.level_4,
        (
            SELECT median(p.netto)
            FROM products_model AS p
            WHERE p.level_4 = m.level_4 AND p.netto IS NOT NULL
        ) AS level_4_median,
        (
            SELECT median(p.netto)
            FROM products_model AS p
            WHERE p.level_3 = m.level_3 AND p.netto IS NOT NULL
        ) AS level_3_median,
        (
            SELECT median(p.netto)
            FROM products_model AS p
            WHERE p.level_2 = m.level_2 AND p.netto IS NOT NULL
        ) AS level_2_median,
        (SELECT median(netto) FROM products_model WHERE netto IS NOT NULL)
            AS global_median
    FROM products_model AS m
    WHERE m.netto IS NULL
    """
).df()

display(netto_imputation_candidates)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,product_id,level_2,level_3,level_4,level_4_median,level_3_median,level_2_median,global_median
0,04d86b4b50,None,None,None,NaN,NaN,NaN,0.3000
1,48cc0e256d,None,None,None,NaN,NaN,NaN,0.3000
2,6a3d708544,None,None,None,NaN,NaN,NaN,0.3000


In [418]:
extreme_netto_values = duckdb_connection.execute(
    """
    WITH threshold AS (
        SELECT quantile_cont(netto, 0.999) AS p999
        FROM products_model
        WHERE netto IS NOT NULL
    )
    SELECT netto, COUNT(*) AS products
    FROM products_model
    CROSS JOIN threshold
    WHERE netto >= p999
    GROUP BY netto
    ORDER BY netto DESC
    """
).df()

netto_display = extreme_netto_values.head(30)

styled = style_table(
    netto_display,
    formats={"netto": "{:,.4f}","products": "{:,.0f}",},
)

max_idx = netto_display.index[netto_display["netto"] == netto_display["netto"].max()]

display(paint(styled,max_idx,["netto"],"blue",))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

netto,products
"1,150.0000",1
600.0000,2
430.0000,1
400.0000,1
350.0000,1
316.8000,2
260.0000,1
259.2000,2
130.0000,1
80.0000,1


In [419]:
extreme_netto_products = duckdb_connection.execute(
    """
    WITH threshold AS (
        SELECT quantile_cont(netto, 0.999) AS p999
        FROM products_model
        WHERE netto IS NOT NULL
    )
    SELECT
        p.product_id,
        p.level_1,
        p.level_2,
        p.level_3,
        p.level_4,
        p.netto
    FROM products_model AS p
    CROSS JOIN threshold
    WHERE p.netto >= threshold.p999
    ORDER BY p.netto DESC
    """
).df()

display(
    style_table(
        extreme_netto_products.head(30),
        {"netto": "{:,.4f}"},
    )
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

product_id,level_1,level_2,level_3,level_4,netto
ec9077783d,c3d3a8e8c6,f93982269d,0bcfc6519b,b4b0e4c470,"1,150.0000"
98528c8b99,c3d3a8e8c6,f93982269d,0bcfc6519b,b4b0e4c470,600.0000
2fbd8c3d9c,c3d3a8e8c6,f93982269d,0bcfc6519b,b4b0e4c470,600.0000
61a26b3d4b,e344ab2e71,52f13dac0c,6b55683dad,56426fcb60,430.0000
0020caa486,e344ab2e71,52f13dac0c,6b55683dad,56426fcb60,400.0000
b46720cd94,c3d3a8e8c6,f93982269d,0bcfc6519b,fc32a80dcd,350.0000
42511cfd14,c3d3a8e8c6,ad2b2e17d2,ca69ed9de2,8bbeabc581,316.8000
44f2e529ac,c3d3a8e8c6,ad2b2e17d2,ca69ed9de2,8bbeabc581,316.8000
50e2016e67,e344ab2e71,52f13dac0c,6b55683dad,56426fcb60,260.0000
d07c11c602,c3d3a8e8c6,ad2b2e17d2,ca69ed9de2,8bbeabc581,259.2000


In [420]:
extreme_netto_purchase_impact = duckdb_connection.execute(
    """
    WITH threshold AS (
        SELECT quantile_cont(netto, 0.999) AS p999
        FROM products_model
        WHERE netto IS NOT NULL
    ),
    extreme_products AS (
        SELECT product_id
        FROM products_model
        CROSS JOIN threshold
        WHERE netto >= p999
    )
    SELECT
        COUNT(DISTINCT ep.product_id) AS extreme_products,
        COUNT(p.product_id) AS purchase_rows,
        COUNT(DISTINCT p.client_id) AS customers
    FROM extreme_products AS ep
    LEFT JOIN purchases_model AS p USING (product_id)
    """
).df()

display(extreme_netto_purchase_impact)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,extreme_products,purchase_rows,customers
0,96,76959,33548


### `netto` Data Quality

For the **40,716 products** relevant to the modeling population:

* Median `netto` is approximately `0.30`
* 95% of products have `netto <= 1.00`
* 99% have `netto <= 1.56`
* 99.9% have `netto <= 5.00`
* The maximum is `1,150`
* Only **3 products** have missing `netto`

The distribution contains unusually large values. However, the available data does not provide enough information about the meaning or unit of `netto` to determine whether these values are incorrect.

Applying clipping or imputation would therefore introduce assumptions that cannot be supported by the data.

### Current Conclusion

* Keep observed `netto` values unchanged in the cleaned product table.
* Keep the three missing values as `NULL`.
* Do not apply clipping, winsorization, or imputation.
* Do not use `netto` for customer feature construction.

### Product Quantity

`product_quantity` records the quantity of a product in a purchase row.

Most values are small, but the distribution has a long upper tail. The checks below look at:

* The overall quantity distribution
* Negative and zero values
* Unusually large quantities
* Quantity patterns for the same product
* Whether zero quantities occur across many products
* Whether quantities are always whole numbers

The goal is to decide whether any quantity values should be changed during cleaning.

In [421]:
#how quickly the upper tail grows
product_quantity_distribution = duckdb_connection.execute(
    """
    SELECT
        COUNT(*) AS purchase_rows,
        MIN(product_quantity) AS min,
        quantile_cont(product_quantity, 0.50) AS median,
        quantile_cont(product_quantity, 0.95) AS p95,
        quantile_cont(product_quantity, 0.99) AS p99,
        quantile_cont(product_quantity, 0.999) AS p999,
        quantile_cont(product_quantity, 0.9999) AS p9999,
        MAX(product_quantity) AS max
    FROM purchases_model
    WHERE product_quantity IS NOT NULL
    """
).df()

display(product_quantity_distribution)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,purchase_rows,min,median,p95,p99,p999,p9999,max
0,22882690,0.0000,1.0000,3.0000,5.0000,11.0000,30.0000,648.0000


In [422]:
#how common zero quantities are
product_quantity_status = duckdb_connection.execute(
    """
    SELECT
        COUNT(*) FILTER (WHERE product_quantity < 0) AS negative_rows,
        COUNT(*) FILTER (WHERE product_quantity = 0) AS zero_rows,
        COUNT(*) FILTER (WHERE product_quantity > 0) AS positive_rows,
        ROUND(100.0 * COUNT(*) FILTER (WHERE product_quantity = 0) / COUNT(*), 4) AS zero_rate
    FROM purchases_model
    WHERE product_quantity IS NOT NULL
    """
).df()

display(product_quantity_status)

,negative_rows,zero_rows,positive_rows,zero_rate
0,0,1354091,21528599,5.9175


In [423]:
#if there is a clear gap or isolated extremes
product_quantity_tail = duckdb_connection.execute(
    """
    WITH threshold AS (
        SELECT quantile_cont(product_quantity, 0.999) AS p999
        FROM purchases_model
        WHERE product_quantity IS NOT NULL
    )
    SELECT product_quantity, COUNT(*) AS purchase_rows
    FROM purchases_model
    CROSS JOIN threshold
    WHERE product_quantity >= p999
    GROUP BY product_quantity
    ORDER BY product_quantity DESC
    LIMIT 30
    """
).df()

display(product_quantity_tail)

,product_quantity,purchase_rows
0,648.0000,1
1,600.0000,1
2,300.0000,1
3,278.0000,1
4,254.0000,1
5,243.0000,1
6,240.0000,1
7,210.0000,1
8,202.0000,1
9,200.0000,3


In [424]:
#if large quantities are normal for specific products.
large_quantity_products = duckdb_connection.execute(
    """
    SELECT
        product_id,
        COUNT(*) AS purchase_rows,
        quantile_cont(product_quantity, 0.50) AS median_quantity,
        quantile_cont(product_quantity, 0.99) AS p99_quantity,
        MAX(product_quantity) AS max_quantity
    FROM purchases_model
    WHERE product_quantity > 0
    GROUP BY product_id
    ORDER BY max_quantity DESC
    LIMIT 30
    """
).df()

display(large_quantity_products)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,product_id,purchase_rows,median_quantity,p99_quantity,max_quantity
0,ee4f7132c3,13868,2.0000,15.0000,648.0000
1,4dcf79043e,185431,1.0000,12.0000,300.0000
2,a755a6f2e6,6891,1.0000,4.0000,278.0000
3,197c432c53,6572,2.0000,10.0000,254.0000
4,f2293d7dfa,8207,2.0000,30.0000,243.0000
5,1c257c1a1b,65929,1.0000,5.0000,210.0000
6,7302b8e550,6553,2.0000,11.0000,202.0000
7,4009f09b04,910569,1.0000,3.0000,200.0000
8,2307c1ad47,11576,2.0000,6.0000,190.0000
9,c581cdb29e,11560,3.0000,12.0000,180.0000


In [425]:
#Are zero quantities random errors, or a systemic pattern across products?
zero_quantity_profile = duckdb_connection.execute(
    """
    WITH product_profile AS (
        SELECT
            product_id,
            COUNT(*) FILTER (WHERE product_quantity = 0) AS zero_rows,
            COUNT(*) FILTER (WHERE product_quantity > 0) AS positive_rows
        FROM purchases_model
        GROUP BY product_id
    )
    SELECT
        COUNT(*) FILTER (WHERE zero_rows > 0) AS products_with_zero,
        COUNT(*) FILTER (WHERE zero_rows > 0 AND positive_rows > 0) AS products_with_zero_and_positive,
        COUNT(*) FILTER (WHERE zero_rows > 0 AND positive_rows = 0) AS zero_only_products,
        SUM(zero_rows) AS zero_rows
    FROM product_profile
    """
).df()

display(zero_quantity_profile)

,products_with_zero,products_with_zero_and_positive,zero_only_products,zero_rows
0,1219,903,316,"1,354,091.0000"


In [426]:
#whole numbers only, or decimals.
quantity_integer_check = duckdb_connection.execute(
    """
    SELECT
        COUNT(*) AS purchase_rows,
        COUNT(*) FILTER (WHERE product_quantity != FLOOR(product_quantity)) AS fractional_rows
    FROM purchases_model
    WHERE product_quantity IS NOT NULL
    """
).df()

display(quantity_integer_check)

,purchase_rows,fractional_rows
0,22882690,0


In [427]:
#details of the largest quantities to distinguish between data errors and valid high-volume purchases.
largest_quantity_rows = duckdb_connection.execute(
    """
    SELECT
        p.client_id,
        p.transaction_id,
        p.transaction_datetime,
        p.product_id,
        p.product_quantity,
        p.purchase_sum,
        p.trn_sum_from_iss,
        pr.level_1,
        pr.level_2,
        pr.level_3,
        pr.level_4,
        pr.netto
    FROM purchases_model AS p
    LEFT JOIN products_model AS pr USING (product_id)
    ORDER BY p.product_quantity DESC
    LIMIT 20
    """
).df()

display(largest_quantity_rows)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_id,transaction_id,transaction_datetime,product_id,product_quantity,purchase_sum,trn_sum_from_iss,level_1,level_2,level_3,level_4,netto
0,d5eda4436b,482a36036c,2019-01-16 07:01:52,ee4f7132c3,648.0000,"25,913.0000","25,914.0000",e344ab2e71,14d373dff5,39532a0f6f,e66f0cae96,0.4500
1,d5eda4436b,5ebb3e6359,2018-12-13 13:33:37,ee4f7132c3,600.0000,"23,994.0000","23,994.0000",e344ab2e71,14d373dff5,39532a0f6f,e66f0cae96,0.4500
2,643b4bd998,1fea866a7b,2019-02-28 06:29:43,4dcf79043e,300.0000,"11,206.0000","11,067.0000",e344ab2e71,ed2ad1797c,c4669106da,9b12b02458,1.0000
3,0f3a42777b,d2211d44a7,2018-12-05 10:29:49,a755a6f2e6,278.0000,"12,924.0600","11,117.0000",e344ab2e71,703f4b6eb0,8a37c27a14,8587e8dfd1,0.0950
4,6d3627f6b7,5ecbf6954c,2018-12-14 15:36:09,197c432c53,254.0000,"1,621.3500",79.0000,c3d3a8e8c6,ad2b2e17d2,ca69ed9de2,9ea7822731,0.0400
5,b23c45b37e,22539561b2,2019-02-08 08:11:02,f2293d7dfa,243.0000,"2,792.0000","2,428.0000",e344ab2e71,703f4b6eb0,f4613d272f,47fc199714,0.0150
6,d5eda4436b,a3bd866e11,2018-12-09 16:45:39,ee4f7132c3,240.0000,"9,597.0000","9,598.0000",e344ab2e71,14d373dff5,39532a0f6f,e66f0cae96,0.4500
7,604edd452c,8f67f78def,2018-12-17 08:35:50,1c257c1a1b,210.0000,"10,203.0000","10,204.0000",e344ab2e71,ed2ad1797c,1f2ed5f7a5,a69b66110a,0.8280
8,bb211cc796,25260b5e03,2018-12-29 06:41:54,7302b8e550,202.0000,"14,575.0000","8,078.0000",e344ab2e71,14d373dff5,39532a0f6f,e66f0cae96,0.4800
9,f04ac8e9bd,0309aec8bf,2019-01-16 12:09:36,4dcf79043e,200.0000,"7,796.0000","7,796.0000",e344ab2e71,ed2ad1797c,c4669106da,9b12b02458,1.0000


### Product Quantity Findings

`product_quantity` is strongly concentrated around small values:

* Median = `1`
* p95 = `3`
* p99 = `5`
* p99.9 = `11`
* p99.99 = `30`
* Maximum = `648`

There are no negative or fractional quantities.

Zero quantities are common enough to form a clear source-data pattern:

* `1,354,091` rows have quantity `0`
* `1,219` products have at least one zero-quantity row
* `903` of these products also have positive quantities
* `316` products appear only with zero quantity

The large quantities are rare, but the detailed records do not show an obvious structural error. For example, the two largest values, `648` and `600`, belong to the same product and customer on different dates, and both occur in transactions with large purchase amounts.

These values are clearly unusual compared with the normal range of the same products, but there is not enough evidence to say that they are incorrect.

### Current Conclusion

* Keep zero quantities
* Keep the observed upper-tail quantities for now
* Do not remove purchase rows based only on quantity

### Purchase Table Grain and Transaction Identity

Before purchase history can be aggregated to customer-level features, we need to understand what one row and one transaction represent.

The checks below answer four questions:

* Does the table contain duplicate rows?
* Is `transaction_id` unique?
* How many product rows belong to one transaction?
* Which columns are transaction-level values and which are product-line values?

Getting this grain right is important because transaction totals must not be counted once for every product row.

In [428]:
purchase_duplicate_check = duckdb_connection.execute(
    """
    WITH distinct_rows AS (
        SELECT DISTINCT *
        FROM purchases_model
    )
    SELECT
        (SELECT COUNT(*) FROM purchases_model) AS purchase_rows,
        (SELECT COUNT(*) FROM distinct_rows) AS distinct_purchase_rows
    """
).df()

purchase_duplicate_check["duplicate_rows"] = (
    purchase_duplicate_check["purchase_rows"] - purchase_duplicate_check["distinct_purchase_rows"]
)

display(purchase_duplicate_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,purchase_rows,distinct_purchase_rows,duplicate_rows
0,22882690,22882690,0


In [429]:
# `transaction_id` alone always identifies one transaction.
reused_transaction_id_check = duckdb_connection.execute(
    """
    WITH transaction_instances AS (
        SELECT DISTINCT transaction_id, client_id, transaction_datetime, store_id
        FROM purchases_model
    ),
    transaction_profile AS (
        SELECT transaction_id, COUNT(*) AS instances
        FROM transaction_instances
        GROUP BY transaction_id
    )
    SELECT
        COUNT(*) AS transaction_ids,
        COUNT(*) FILTER (WHERE instances > 1) AS reused_transaction_ids,
        MAX(instances) AS max_instances_per_id
    FROM transaction_profile
    """
).df()

display(reused_transaction_id_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,transaction_ids,reused_transaction_ids,max_instances_per_id
0,4024941,8,2


In [430]:
#picture of the size of purchase invoices
transaction_grain_profile = duckdb_connection.execute(
    """
    WITH transactions AS (
        SELECT
            transaction_id, client_id, transaction_datetime, store_id,
            COUNT(*) AS rows_per_transaction,
            COUNT(DISTINCT product_id) AS products_per_transaction
        FROM purchases_model
        GROUP BY transaction_id, client_id, transaction_datetime, store_id
    )
    SELECT
        COUNT(*) AS transactions,
        quantile_cont(rows_per_transaction, 0.50) AS median_rows,
        quantile_cont(rows_per_transaction, 0.95) AS p95_rows,
        quantile_cont(rows_per_transaction, 0.99) AS p99_rows,
        MAX(rows_per_transaction) AS max_rows,
        MAX(products_per_transaction) AS max_products
    FROM transactions
    """
).df()

display(transaction_grain_profile)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,transactions,median_rows,p95_rows,p99_rows,max_rows,max_products
0,4024949,4.0000,15.0000,24.0000,116,116


In [431]:
#product_id appears more than once in a transaction_id.
duplicate_product_check = duckdb_connection.execute(
    """
    WITH transaction_products AS (
        SELECT
            transaction_id, client_id, transaction_datetime, store_id, product_id,
            COUNT(*) AS rows
        FROM purchases_model
        GROUP BY transaction_id, client_id, transaction_datetime, store_id, product_id
    )
    SELECT
        COUNT(*) FILTER (WHERE rows > 1) AS duplicated_transaction_product_pairs,
        COALESCE(SUM(rows - 1) FILTER (WHERE rows > 1), 0) AS extra_duplicate_rows
    FROM transaction_products
    """
).df()

display(duplicate_product_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicated_transaction_product_pairs,extra_duplicate_rows
0,0,0.0000


In [432]:
#the total amount and reward points of an order are not skewed between product lines
transaction_field_consistency = duckdb_connection.execute(
    """
    WITH transactions AS (
        SELECT
            transaction_id, client_id, transaction_datetime, store_id,
            COUNT(DISTINCT purchase_sum) AS purchase_sum_values,
            COUNT(DISTINCT regular_points_received) AS regular_received_values,
            COUNT(DISTINCT express_points_received) AS express_received_values,
            COUNT(DISTINCT regular_points_spent) AS regular_spent_values,
            COUNT(DISTINCT express_points_spent) AS express_spent_values
        FROM purchases_model
        GROUP BY transaction_id, client_id, transaction_datetime, store_id
    )
    SELECT
        COUNT(*) FILTER (WHERE purchase_sum_values > 1) AS inconsistent_purchase_sum,
        COUNT(*) FILTER (WHERE regular_received_values > 1) AS inconsistent_regular_received,
        COUNT(*) FILTER (WHERE express_received_values > 1) AS inconsistent_express_received,
        COUNT(*) FILTER (WHERE regular_spent_values > 1) AS inconsistent_regular_spent,
        COUNT(*) FILTER (WHERE express_spent_values > 1) AS inconsistent_express_spent
    FROM transactions
    """
).df()

display(transaction_field_consistency)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,inconsistent_purchase_sum,inconsistent_regular_received,inconsistent_express_received,inconsistent_regular_spent,inconsistent_express_spent
0,0,0,0,0,0


In [433]:
#trn_sum_from_iss and trn_sum_from_red values ​​change between products in the same invoice
line_field_profile = duckdb_connection.execute(
    """
    WITH transactions AS (
        SELECT
            transaction_id, client_id, transaction_datetime, store_id,
            COUNT(*) AS rows,
            COUNT(DISTINCT trn_sum_from_iss) AS iss_values,
            COUNT(DISTINCT trn_sum_from_red) AS red_values
        FROM purchases_model
        GROUP BY transaction_id, client_id, transaction_datetime, store_id
    )
    SELECT
        COUNT(*) FILTER (WHERE rows > 1) AS multirow_transactions,
        COUNT(*) FILTER (WHERE rows > 1 AND iss_values > 1) AS transactions_with_varying_iss,
        COUNT(*) FILTER (WHERE rows > 1 AND red_values > 1) AS transactions_with_varying_red
    FROM transactions
    """
).df()

display(line_field_profile)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,multirow_transactions,transactions_with_varying_iss,transactions_with_varying_red
0,3490044,3454548,208369


In [434]:
#total purchase_sum with the cumulative total trn_sum_from_iss to see if they match.
iss_purchase_relation = duckdb_connection.execute(
    """
    WITH transactions AS (
        SELECT
            transaction_id, client_id, transaction_datetime, store_id,
            MAX(purchase_sum) AS purchase_sum,
            SUM(trn_sum_from_iss) AS iss_sum
        FROM purchases_model
        GROUP BY transaction_id, client_id, transaction_datetime, store_id
    )
    SELECT
        quantile_cont(ABS(iss_sum - purchase_sum), 0.50) AS median_absolute_difference,
        quantile_cont(ABS(iss_sum - purchase_sum), 0.95) AS p95_absolute_difference,
        ROUND(
            100.0 * COUNT(*) FILTER (WHERE ABS(iss_sum - purchase_sum) < 0.01) / COUNT(*),
            2
        ) AS pct_near_exact_match
    FROM transactions
    """
).df()

display(iss_purchase_relation)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,median_absolute_difference,p95_absolute_difference,pct_near_exact_match
0,0.6400,67.0000,19.6100


In [435]:
#customers in uplift_train actually have a purchase history.
purchase_history_coverage = duckdb_connection.execute(
    """
    WITH history AS (
        SELECT client_id, COUNT(*) AS purchase_rows
        FROM purchases_model
        GROUP BY client_id
    )
    SELECT
        COUNT(*) AS modeling_customers,
        COUNT(h.client_id) AS customers_with_purchase_history,
        COUNT(*) FILTER (WHERE h.client_id IS NULL) AS customers_without_purchase_history,
        MIN(h.purchase_rows) AS min_purchase_rows,
        quantile_cont(h.purchase_rows, 0.50) AS median_purchase_rows,
        MAX(h.purchase_rows) AS max_purchase_rows
    FROM clients_model AS c
    LEFT JOIN history AS h USING (client_id)
    """
).df()

display(purchase_history_coverage)

,modeling_customers,customers_with_purchase_history,customers_without_purchase_history,min_purchase_rows,median_purchase_rows,max_purchase_rows
0,200039,200039,0,1,85.0000,2247


#### Purchase-History Coverage Result

All **200,039 modeling customers** have purchase history.

The number of purchase rows per customer ranges from `1` to `2,247`, with a median of `85`.

In [436]:
#first_issue_date or first_redeem_date that is even after the end date of their purchase history.
customer_date_temporal_check = duckdb_connection.execute(
    """
    WITH history AS (
        SELECT MAX(transaction_datetime) AS purchase_history_end
        FROM purchases_model
    )
    SELECT
        purchase_history_end,
        COUNT(*) FILTER (WHERE first_issue_date > purchase_history_end) AS issue_after_history_end,
        COUNT(*) FILTER (WHERE first_redeem_date > purchase_history_end) AS redeem_after_history_end
    FROM clients_model
    CROSS JOIN history
    GROUP BY purchase_history_end
    """
).df()

display(customer_date_temporal_check)

,purchase_history_end,issue_after_history_end,redeem_after_history_end
0,2019-03-18 23:19:28,0,22600


#### Temporal Coverage Result

The available purchase history ends on **2019-03-18 23:19:28**.

No customer has `first_issue_date` after this point.

However, **22,600 customers** have `first_redeem_date` after the end of the purchase-history window.

These redemption dates are not automatically treated as data errors. They may still be valid customer records, but they should not be used blindly as pre-treatment features. Their timing should be checked again during feature engineering.

### Purchase Table Grain Findings

* **Dataset Size:** Contains **22,882,690 purchase rows** and **4,024,949 transactions** with no exact duplicate rows.
* **Transaction Key:** `transaction_id` is reused 8 times, so a unique transaction must be identified by combining four columns: 
  `transaction_id + client_id + transaction_datetime + store_id`.
* **Data Structure Levels:**
  * **Transaction Level:** `purchase_sum` and loyalty points are constant across all rows of the same order.
  * **Product-Line Level:** `product_id`, `product_quantity`, `trn_sum_from_iss`, and `trn_sum_from_red` vary by item.
* **Key Warning:** Do not treat `trn_sum_from_iss` as the total purchase amount. Their sums only match in about **19.61%** of transactions.

### Current Conclusion

* Keep all original purchase rows.
* Always use the 4-column composite key to group and analyze transactions.
* Count transaction-level totals (like `purchase_sum` and points) **only once per transaction** to avoid overcounting.
* Keep `trn_sum_*` columns separate from `purchase_sum`.

#### Transaction-Level View

In [437]:
duckdb_connection.execute(
    """
    CREATE OR REPLACE VIEW transactions_model AS
    SELECT
        transaction_id,
        client_id,
        transaction_datetime,
        store_id,
        MAX(purchase_sum) AS purchase_sum,
        MAX(regular_points_received) AS regular_points_received,
        MAX(express_points_received) AS express_points_received,
        MAX(regular_points_spent) AS regular_points_spent,
        MAX(express_points_spent) AS express_points_spent
    FROM purchases_model
    GROUP BY transaction_id, client_id, transaction_datetime, store_id
    """
)

In [438]:
transaction_numeric_profile = duckdb_connection.execute(
    """
    SUMMARIZE
    SELECT
        purchase_sum,
        regular_points_received,
        express_points_received,
        regular_points_spent,
        express_points_spent
    FROM transactions_model
    """
).df()

display(transaction_numeric_profile)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,purchase_sum,DOUBLE,0.0,29611.48,230638,427.29477920834523,487.40163934994166,140.50109493078412,282.0027827950985,532.7386012918641,4024949,0.0000
1,regular_points_received,DOUBLE,0.0,2399.0,2077,3.8777185499737747,7.897550589686727,0.6,1.3628052120654914,3.886030497120065,4024949,0.0000
2,express_points_received,DOUBLE,0.0,300.0,12,0.03959677501503746,1.7129696596403643,0.0,0.0,0.0,4024949,0.0000
3,regular_points_spent,DOUBLE,-5066.0,0.0,962,-3.647817649366489,25.769722635549794,0.0,0.0,0.0,4024949,0.0000
4,express_points_spent,DOUBLE,-300.0,0.0,63,-0.32077872291052634,3.247379571647986,0.0,0.0,0.0,4024949,0.0000


In [439]:
line_numeric_profile = duckdb_connection.execute(
    """
    SUMMARIZE
    SELECT trn_sum_from_iss, trn_sum_from_red
    FROM purchases_model
    """
).df()

display(line_numeric_profile)

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,trn_sum_from_iss,DOUBLE,0.0,29074.0,3094,73.42164579426633,86.63618106525328,30.016384217123026,50.9713676738505,89.84113260178857,22882690,0.0000
1,trn_sum_from_red,DOUBLE,0.0,6400.0,1496,76.781495536201,84.5437286267354,31.22576933867122,54.516999506291135,94.11824589952526,22882690,93.3900


In [440]:
#negative reward points (focus) with transactions containing a trn_sum_from_red value
redemption_relation = duckdb_connection.execute(
    """
    WITH transactions AS (
        SELECT
            transaction_id, client_id, transaction_datetime, store_id,
            MAX(regular_points_spent) AS regular_points_spent,
            MAX(express_points_spent) AS express_points_spent,
            COUNT(trn_sum_from_red) AS red_rows
        FROM purchases_model
        GROUP BY transaction_id, client_id, transaction_datetime, store_id
    )
    SELECT
        COUNT(*) AS transactions,
        COUNT(*) FILTER (
            WHERE regular_points_spent < 0 OR express_points_spent < 0
        ) AS transactions_with_points_spent,
        COUNT(*) FILTER (WHERE red_rows > 0) AS transactions_with_trn_sum_from_red,
        COUNT(*) FILTER (
            WHERE (regular_points_spent < 0 OR express_points_spent < 0) AND red_rows > 0
        ) AS points_spent_and_trn_sum_from_red,
        COUNT(*) FILTER (
            WHERE (regular_points_spent < 0 OR express_points_spent < 0) AND red_rows = 0
        ) AS points_spent_without_trn_sum_from_red,
        COUNT(*) FILTER (
            WHERE regular_points_spent = 0 AND express_points_spent = 0 AND red_rows > 0
        ) AS trn_sum_from_red_without_points_spent
    FROM transactions
    """
).df()

display(redemption_relation)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,transactions,transactions_with_points_spent,transactions_with_trn_sum_from_red,points_spent_and_trn_sum_from_red,points_spent_without_trn_sum_from_red,trn_sum_from_red_without_points_spent
0,4024949,251609,252467,251606,3,861


In [441]:
#trn_sum_from_red has a very high missing rate
redemption_missing_profile = duckdb_connection.execute(
    """
    SELECT
        COUNT(*) AS purchase_rows,
        COUNT(*) FILTER (WHERE trn_sum_from_red IS NULL) AS missing_rows,
        COUNT(*) FILTER (WHERE trn_sum_from_red IS NOT NULL) AS nonmissing_rows,
        ROUND(
            100.0 * COUNT(*) FILTER (WHERE trn_sum_from_red IS NULL) / COUNT(*),
            2
        ) AS missing_rate,
        COUNT(*) FILTER (WHERE trn_sum_from_red = 0) AS zero_rows,
        COUNT(*) FILTER (WHERE trn_sum_from_red > 0) AS positive_rows
    FROM purchases_model
    """
).df()

display(redemption_missing_profile)

,purchase_rows,missing_rows,nonmissing_rows,missing_rate,zero_rows,positive_rows
0,22882690,21369293,1513397,93.3900,17322,1496075



* Transaction-level totals and points received are non-negative, while points spent use negative values correctly.
* `trn_sum_from_iss` is available everywhere, but `trn_sum_from_red` is missing in **93.39%** of rows.
* `trn_sum_from_red` and point spending are closely linked, but they are not identical signals:
  * Over 251,000 transactions contain both.
  * A tiny fraction have one without the other (e.g., 3 transactions have points spent without `trn_sum_from_red`, and 861 have it without points spent).

### Current Conclusion

* Keep all original monetary values and point signs as they are.
* Retain both `trn_sum_iss` and `trn_sum_red` columns for now.

### Product Metadata Structure

The product table contains category levels together with brand, segment, vendor, and other product attributes.

The checks below focus on:

* How many distinct values each metadata field contains;
* Whether `level_1`–`level_4` form a consistent hierarchy;
* Whether `segment_id` behaves like an identifier rather than a continuous number;
* Whether missing category levels occur together or as partial gaps.


In [442]:
product_metadata_cardinality = duckdb_connection.execute(
    """
    SELECT
        COUNT(*) AS products,
        COUNT(DISTINCT brand_id) AS brands,
        COUNT(DISTINCT segment_id) AS segments,
        COUNT(DISTINCT vendor_id) AS vendors,
        COUNT(DISTINCT level_1) AS level_1_values,
        COUNT(DISTINCT level_2) AS level_2_values,
        COUNT(DISTINCT level_3) AS level_3_values,
        COUNT(DISTINCT level_4) AS level_4_values
    FROM products_model
    """
).df()

display(product_metadata_cardinality)

,products,brands,segments,vendors,level_1_values,level_2_values,level_3_values,level_4_values
0,40716,4183,116,3136,3,42,201,783


In [443]:
product_hierarchy_consistency = duckdb_connection.execute(
    """
    SELECT
        'level_2 -> level_1' AS relationship,
        COUNT(*) FILTER (WHERE parent_count > 1) AS children_with_multiple_parents
    FROM (
        SELECT level_2, COUNT(DISTINCT level_1) AS parent_count
        FROM products_model
        WHERE level_1 IS NOT NULL AND level_2 IS NOT NULL
        GROUP BY level_2
    )

    UNION ALL

    SELECT
        'level_3 -> level_2',
        COUNT(*) FILTER (WHERE parent_count > 1)
    FROM (
        SELECT level_3, COUNT(DISTINCT level_2) AS parent_count
        FROM products_model
        WHERE level_2 IS NOT NULL AND level_3 IS NOT NULL
        GROUP BY level_3
    )

    UNION ALL

    SELECT
        'level_4 -> level_3',
        COUNT(*) FILTER (WHERE parent_count > 1)
    FROM (
        SELECT level_4, COUNT(DISTINCT level_3) AS parent_count
        FROM products_model
        WHERE level_3 IS NOT NULL AND level_4 IS NOT NULL
        GROUP BY level_4
    )
    """
).df()

display(product_hierarchy_consistency)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,relationship,children_with_multiple_parents
0,level_2 -> level_1,0
1,level_3 -> level_2,0
2,level_4 -> level_3,0


In [444]:
#contains a decimal number
segment_id_check = duckdb_connection.execute(
    """
    SELECT
        COUNT(*) FILTER (
            WHERE segment_id IS NOT NULL
              AND segment_id != FLOOR(segment_id)
        ) AS fractional_segment_ids
    FROM products_model
    """
).df()

display(segment_id_check)

,fractional_segment_ids
0,0


In [445]:
# lost its category information in a partial or total loss manner
product_metadata_missing_pattern = duckdb_connection.execute(
    """
    SELECT
        COUNT(*) FILTER (
            WHERE level_1 IS NULL AND level_2 IS NULL
              AND level_3 IS NULL AND level_4 IS NULL
        ) AS all_hierarchy_levels_missing,

        COUNT(*) FILTER (
            WHERE (level_1 IS NULL OR level_2 IS NULL OR level_3 IS NULL OR level_4 IS NULL)
              AND NOT (
                  level_1 IS NULL AND level_2 IS NULL
                  AND level_3 IS NULL AND level_4 IS NULL
              )
        ) AS partial_hierarchy_missing,

        COUNT(*) FILTER (WHERE brand_id IS NULL) AS missing_brand,
        COUNT(*) FILTER (WHERE segment_id IS NULL) AS missing_segment,
        COUNT(*) FILTER (WHERE vendor_id IS NULL) AS missing_vendor
    FROM products_model
    """
).df()

display(product_metadata_missing_pattern)

,all_hierarchy_levels_missing,partial_hierarchy_missing,missing_brand,missing_segment,missing_vendor
0,3,0,4767,1457,26


### Product Metadata Findings

The **40,716 products** in the modeling data include:

* `3` level-1 categories
* `42` level-2 categories
* `201` level-3 categories
* `783` level-4 categories
* `4,183` brands
* `116` segments
* `3,136` vendors

The category hierarchy is fully consistent:

`level_2 → level_1`

`level_3 → level_2`

`level_4 → level_3`

No lower-level category maps to more than one parent.

`segment_id` contains no fractional values, confirming it acts as an identifier instead of a continuous numeric measure.

Missing hierarchy values follow a distinct pattern:

* `3` products lack all four hierarchy levels entirely
* There are **no partial hierarchy gaps**
* `4,767` products are missing `brand_id`
* `1,457` are missing `segment_id`
* `26` are missing `vendor_id`

Because those three products miss every category level, their categories cannot be recovered using the hierarchy alone.

### Current Conclusion

The category hierarchy is clean and will be kept as provided.

`segment_id`, `brand_id`, and `vendor_id` should be treated as product identifiers rather than continuous numbers.

## Missing-Value Handling

Missing values are handled based on what can be recovered from the data. The goal is to keep useful records without adding unsupported values.

### Customer Data

* **`age`:** converted to `NULL`. 
* **`first_redeem_date`:** Existing missing values remain `NULL`. The single cross-day date inconsistency will also be converted to `NULL`. 

### Product Data

* **`level_1`–`level_4`:** Since no category level is available to recover these values, the missing categories are represented as `UNKNOWN`.

* **`brand_id`, `segment_id`, `vendor_id`, :** `4,767` These values are represented as `UNKNOWN`.

* **`netto`:** Their missing `netto` values are replaced with the median `netto` of the relevant product population.

### Purchase Data

* **`trn_sum_from_red`:** 93.39% of values are missing. 251,606 transactions have both points spent and trn_sum_from_red, while only 3 have points spent without it and 861 have trn_sum_from_red without points spent. Therefore, this column is dropped.

* **Zero `product_quantity`:** Zero is an observed value, not a missing value, and is therefore kept unchanged.

Any missing-value indicators or model-specific imputation can be added later during feature engineering if needed.

## Data Cleaning

In [446]:
duckdb_connection.execute(
    """
    CREATE OR REPLACE VIEW clients_clean AS
    SELECT * REPLACE (
        CASE WHEN age BETWEEN 13 AND 100 THEN age ELSE NULL END AS age,
        CASE
            WHEN first_redeem_date IS NULL THEN NULL
            WHEN first_redeem_date < first_issue_date
             AND CAST(first_redeem_date AS DATE) = CAST(first_issue_date AS DATE)
                THEN first_issue_date
            WHEN first_redeem_date < first_issue_date THEN NULL
            ELSE first_redeem_date
        END AS first_redeem_date
    )
    FROM clients_model
    """
)

In [447]:
duckdb_connection.execute(
    """
    CREATE OR REPLACE VIEW products_clean AS
    SELECT * REPLACE (
        COALESCE(level_1, 'UNKNOWN') AS level_1,
        COALESCE(level_2, 'UNKNOWN') AS level_2,
        COALESCE(level_3, 'UNKNOWN') AS level_3,
        COALESCE(level_4, 'UNKNOWN') AS level_4,
        COALESCE(CAST(CAST(segment_id AS BIGINT) AS VARCHAR), 'UNKNOWN') AS segment_id,
        COALESCE(brand_id, 'UNKNOWN') AS brand_id,
        COALESCE(vendor_id, 'UNKNOWN') AS vendor_id,
        COALESCE(
            netto,
            (SELECT median(netto) FROM products_model WHERE netto IS NOT NULL)
        ) AS netto
    )
    FROM products_model
    """
)

In [448]:
duckdb_connection.execute(
    """
    CREATE OR REPLACE VIEW purchases_clean AS
    SELECT * EXCLUDE (trn_sum_from_red)
    FROM purchases_model
    """
)

In [449]:
duckdb_connection.execute(
    """
    CREATE OR REPLACE VIEW uplift_train_clean AS
    SELECT *
    FROM uplift_train_raw
    """
)

In [450]:
cleaning_check = duckdb_connection.execute(
    """
    SELECT
        (SELECT COUNT(*) FROM clients_clean) AS clients,
        (SELECT COUNT(*) FROM products_clean) AS products,
        (SELECT COUNT(*) FROM purchases_clean) AS purchases,
        (SELECT COUNT(*) FROM uplift_train_clean) AS uplift_train
    """
).df()

display(cleaning_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,clients,products,purchases,uplift_train
0,200039,40716,22882690,200039


In [451]:
missing_after_cleaning = duckdb_connection.execute(
    """
    SELECT
        (SELECT COUNT(*) FROM clients_clean
         WHERE age IS NOT NULL AND (age < 13 OR age > 100)) AS invalid_age,
        (SELECT COUNT(*) FROM clients_clean
         WHERE first_redeem_date < first_issue_date) AS invalid_dates,
        (SELECT COUNT(*) FROM products_clean
         WHERE netto IS NULL) AS missing_netto,
        (SELECT COUNT(*) FROM products_clean
         WHERE level_1 IS NULL OR level_2 IS NULL OR level_3 IS NULL OR level_4 IS NULL
            OR segment_id IS NULL OR brand_id IS NULL OR vendor_id IS NULL) AS missing_product_metadata
    """
).df()

display(missing_after_cleaning)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,invalid_age,invalid_dates,missing_netto,missing_product_metadata
0,0,0,0,0


In [452]:
CLEAN_DATA_DIR = INTERIM_DATA_DIR / "clean"
CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)

clean_relations = {
    "clients": "clients_clean",
    "products": "products_clean",
    "purchases": "purchases_clean",
    "uplift_train": "uplift_train_clean",
}

for name, relation in clean_relations.items():
    output_path = CLEAN_DATA_DIR / f"{name}_clean.parquet"

    if output_path.exists():
        output_path.unlink()

    duckdb_connection.execute(
        f"""
        COPY {relation}
        TO '{to_sql_path(output_path)}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
        """
    )

    print(name, "->", output_path)

clients -> d:\thao\d\uplif_model\uplif_customer_selection\data\interim\retailhero\clean\clients_clean.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

products -> d:\thao\d\uplif_model\uplif_customer_selection\data\interim\retailhero\clean\products_clean.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

purchases -> d:\thao\d\uplif_model\uplif_customer_selection\data\interim\retailhero\clean\purchases_clean.parquet
uplift_train -> d:\thao\d\uplif_model\uplif_customer_selection\data\interim\retailhero\clean\uplift_train_clean.parquet
